# Step 1: Data Understanding

This notebook prepares the shared dataset split and preprocessing flow for all later experiments.

Roadmap coverage:
- load the Breast Cancer Wisconsin dataset
- inspect shape, classes, missing values and descriptive statistics
- visualize class balance and target-related correlations
- apply stratified train/validation/test split
- standardize features using only the training split
- save summary tables and figures for later notebooks and the final report

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from data_utils import (
    class_distribution,
    create_stratified_splits,
    load_breast_cancer_dataframe,
    scaled_feature_check,
    split_summary,
    summarize_dataframe,
    top_target_correlations,
)
from project_config import DATASET_NAME, RANDOM_SEED, TEST_SIZE, VALIDATION_SIZE, set_global_seed

set_global_seed()
plt.style.use("ggplot")

OUTPUT_FIGURES = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATASET_NAME}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Configured split ratios -> test: {TEST_SIZE:.2f}, validation: {VALIDATION_SIZE:.2f}")

## Load Dataset and Build Core Tables

In [ ]:
X, y, target_names = load_breast_cancer_dataframe()

dataset_summary = summarize_dataframe(X, y)
class_summary = class_distribution(y, target_names)
correlation_summary = top_target_correlations(X, y, n=10)
missing_summary = X.isna().sum().sort_values(ascending=False).rename("missing_values").reset_index()
missing_summary.columns = ["feature", "missing_values"]
descriptive_stats = X.describe().T[["mean", "std", "min", "max"]].sort_index()

dataset_summary.to_csv(OUTPUT_TABLES / "dataset_summary.csv", index=False)
class_summary.to_csv(OUTPUT_TABLES / "class_distribution.csv", index=False)
correlation_summary.to_csv(OUTPUT_TABLES / "top_target_correlations.csv", index=False)
missing_summary.to_csv(OUTPUT_TABLES / "missing_values.csv", index=False)
descriptive_stats.to_csv(OUTPUT_TABLES / "descriptive_statistics.csv")

display(dataset_summary)
display(class_summary)
display(correlation_summary)

In [ ]:
print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"Total missing values: {int(X.isna().sum().sum())}")
print(f"Duplicate feature rows: {int(X.duplicated().sum())}")

display(missing_summary.head(10))
display(descriptive_stats.head(10))

## Visual Inspection

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(class_summary["class_name"], class_summary["count"], color=["#c44e52", "#4c72b0"])
ax.set_title("Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Sample Count")
for idx, row in class_summary.iterrows():
    ax.text(idx, row["count"] + 5, f"{row['ratio']:.1%}", ha="center")
fig.tight_layout()
fig.savefig(OUTPUT_FIGURES / "class_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
top_features = correlation_summary["feature"].tolist()
corr_frame = X[top_features].assign(target=y)

corr_matrix = corr_frame.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=90)
ax.set_yticks(range(len(corr_matrix.index)))
ax.set_yticklabels(corr_matrix.index)
ax.set_title("Correlation Heatmap for Top Target-Related Features")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(OUTPUT_FIGURES / "top_feature_correlation_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

## Preprocessing Pipeline

In [ ]:
split = create_stratified_splits()
split_table = split_summary(split)
scaling_check = scaled_feature_check(split)

split_table.to_csv(OUTPUT_TABLES / "split_summary.csv", index=False)
scaling_check.to_csv(OUTPUT_TABLES / "scaling_check.csv", index=False)

display(split_table)
display(scaling_check)

print("Train shape:", split.X_train_raw.shape)
print("Validation shape:", split.X_val_raw.shape)
print("Test shape:", split.X_test_raw.shape)

In [ ]:
feature_preview = pd.concat(
    [
        split.X_train_raw.head(3).reset_index(drop=True),
        split.X_train_scaled.head(3).reset_index(drop=True).add_prefix("scaled_"),
    ],
    axis=1,
)
display(feature_preview.iloc[:, :8])

## Technical Notes

- The dataset contains 569 samples and 30 numeric features, which makes it appropriate for both a strong logistic regression baseline and controlled MLP experiments.
- No missing values are present, so the preprocessing pipeline stays focused on split discipline and feature scaling instead of imputation.
- Stratified splitting keeps malignant/benign balance consistent across train, validation and test sets; this is required for fair model comparison.
- Standardization is fit only on the training data and then applied to validation/test sets, preventing information leakage.
- The tables saved under `outputs/tables/` and figures saved under `outputs/figures/` will be reused in later notebooks and the final report.